In [1]:
import boto3
import time

rds_client = boto3.client('rds',  region_name='ap-south-1')

response = rds_client.create_db_instance(
    DBName='feedbackdb',
    DBInstanceIdentifier='feedback-instance',
    AllocatedStorage=20,
    DBInstanceClass='db.t4g.micro',
    Engine='mysql',
    MasterUsername='postgres',
    MasterUserPassword='postgres',
    VpcSecurityGroupIds=['sg-0006b3cf7416c4a30'],
    MultiAZ=False,
    EngineVersion='8.0.35',
    StorageType='gp3',  
    PubliclyAccessible=True
)

print("Creating RDS instance...")
while True:
    response = rds_client.describe_db_instances(DBInstanceIdentifier='feedback-instance')
    status = response['DBInstances'][0]['DBInstanceStatus']
    print(f"Current status: {status}")
    if status == 'available':
        break
    time.sleep(30)

endpoint = response['DBInstances'][0]['Endpoint']['Address']
security_group_id = response['DBInstances'][0]['VpcSecurityGroups'][0]['VpcSecurityGroupId']
print(f"RDS instance is available. Endpoint: {endpoint}, Security Group ID: {security_group_id}")


Creating RDS instance...
Current status: creating
Current status: creating
Current status: creating
Current status: creating
Current status: creating
Current status: creating
Current status: backing-up
Current status: backing-up
Current status: backing-up
Current status: backing-up
Current status: backing-up
Current status: backing-up
Current status: available
RDS instance is available. Endpoint: feedback-instance.c7mc40cu0d20.ap-south-1.rds.amazonaws.com, Security Group ID: sg-0006b3cf7416c4a30


In [4]:
import mysql.connector

connection = mysql.connector.connect(
    host='feedback-instance.c7mc40cu0d20.ap-south-1.rds.amazonaws.com',
    user='postgres',
    password='postgres',
    database='feedbackdb'
)

cursor = connection.cursor()

create_table_query = """
CREATE TABLE feedback (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    email VARCHAR(255) NOT NULL,
    message TEXT NOT NULL
)
"""
cursor.execute(create_table_query)
connection.commit()
print("Feedback table created successfully.")

cursor.close()
connection.close()


ProgrammingError: 1050 (42S01): Table 'feedback' already exists